In [4]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (log_loss, roc_auc_score, f1_score, 
                            accuracy_score, precision_score, recall_score, 
                            matthews_corrcoef, confusion_matrix)
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

In [5]:
X_train = pd.read_csv('../datasets/preprocessed/X_train2.csv')
y_train = pd.read_csv('../datasets/preprocessed/y_train2.csv').squeeze()  # Series 변환
X_val = pd.read_csv('../datasets/preprocessed/X_val2.csv')
y_val = pd.read_csv('../datasets/preprocessed/y_val2.csv').squeeze()

In [6]:
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights_dict = {cls: weight for cls, weight in zip(classes, class_weights)}
print(f"Class weights: {class_weights_dict}")

Class weights: {0: 0.9912868632707775, 1: 1.0088676671214187}


In [7]:
def calculate_main_metric(y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_proba)
    return (acc + f1 + auc) / 3

In [8]:
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
        'depth': trial.suggest_int('depth', 5, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 0.1, 10),
        'border_count': trial.suggest_int('border_count', 64, 255),
        'grow_policy': trial.suggest_categorical('grow_policy', ['SymmetricTree', 'Depthwise']),
        'bootstrap_type': trial.suggest_categorical('bootstrap_type', ['Bayesian', 'Bernoulli']),
        'eval_metric': 'AUC', 
        'early_stopping_rounds': 100,
        'verbose': False,
        'random_seed': 42,
        'auto_class_weights': 'Balanced'  
    }
    
    if params['bootstrap_type'] == 'Bayesian':
        params['bagging_temperature'] = trial.suggest_float('bagging_temperature', 0.0, 10.0)
    elif params['bootstrap_type'] == 'Bernoulli':
        params['subsample'] = trial.suggest_float('subsample', 0.5, 1.0)
    
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = []
    
    for train_idx, val_idx in kf.split(X_train, y_train):
        X_fold_train, y_fold_train = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_fold_val, y_fold_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
        
        model = CatBoostClassifier(**params)
        model.fit(
            X_fold_train, y_fold_train,
            eval_set=(X_fold_val, y_fold_val),
            cat_features=list(X_train.select_dtypes(include='object').columns),
            use_best_model=True
        )
        
        y_proba = model.predict_proba(X_fold_val)[:, 1]
        main_metric = calculate_main_metric(y_fold_val, y_proba)
        cv_scores.append(main_metric)
    
    return np.mean(cv_scores)

In [9]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100, timeout=7200)

[I 2025-05-30 15:48:41,769] A new study created in memory with name: no-name-b2f26f3b-cc4f-4062-9fe0-f66d05f1df33
[I 2025-05-30 15:49:19,893] Trial 0 finished with value: 0.662673742193114 and parameters: {'iterations': 971, 'learning_rate': 0.010329746015987481, 'depth': 10, 'l2_leaf_reg': 0.526221239397788, 'random_strength': 1.0822113397159978, 'border_count': 64, 'grow_policy': 'SymmetricTree', 'bootstrap_type': 'Bayesian', 'bagging_temperature': 5.943939162845439}. Best is trial 0 with value: 0.662673742193114.
[I 2025-05-30 15:49:26,901] Trial 1 finished with value: 0.6808376407764203 and parameters: {'iterations': 524, 'learning_rate': 0.06236325328436855, 'depth': 5, 'l2_leaf_reg': 2.818200243381014, 'random_strength': 7.99170419644066, 'border_count': 139, 'grow_policy': 'Depthwise', 'bootstrap_type': 'Bernoulli', 'subsample': 0.7612026704405604}. Best is trial 1 with value: 0.6808376407764203.
[I 2025-05-30 15:49:31,548] Trial 2 finished with value: 0.6652329768635519 and par

In [10]:
best_params = study.best_params
best_params.update({
    'eval_metric': 'AUC',
    'early_stopping_rounds': 100,
    'random_seed': 42,
    'auto_class_weights': 'Balanced'
})

In [11]:
final_model = CatBoostClassifier(**best_params)
final_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    cat_features=list(X_train.select_dtypes(include='object').columns),
    use_best_model=True
)

0:	test: 0.6108200	best: 0.6108200 (0)	total: 8.3ms	remaining: 13.1s
1:	test: 0.6480369	best: 0.6480369 (1)	total: 16.5ms	remaining: 12.9s
2:	test: 0.6586350	best: 0.6586350 (2)	total: 24.3ms	remaining: 12.8s
3:	test: 0.6545350	best: 0.6586350 (2)	total: 32.8ms	remaining: 12.9s
4:	test: 0.6611169	best: 0.6611169 (4)	total: 40.1ms	remaining: 12.6s
5:	test: 0.6612651	best: 0.6612651 (5)	total: 47.6ms	remaining: 12.5s
6:	test: 0.6638012	best: 0.6638012 (6)	total: 55.1ms	remaining: 12.3s
7:	test: 0.6677928	best: 0.6677928 (7)	total: 63.2ms	remaining: 12.4s
8:	test: 0.6696257	best: 0.6696257 (8)	total: 70.7ms	remaining: 12.3s
9:	test: 0.6697701	best: 0.6697701 (9)	total: 79.1ms	remaining: 12.4s
10:	test: 0.6709538	best: 0.6709538 (10)	total: 86.6ms	remaining: 12.3s
11:	test: 0.6720260	best: 0.6720260 (11)	total: 94.4ms	remaining: 12.3s
12:	test: 0.6720763	best: 0.6720763 (12)	total: 102ms	remaining: 12.3s
13:	test: 0.6728133	best: 0.6728133 (13)	total: 110ms	remaining: 12.3s
14:	test: 0.673

In [12]:
def main_metric_optimized_threshold(model, X_val, y_val):
    y_proba = model.predict_proba(X_val)[:, 1]
    thresholds = np.linspace(0.2, 0.8, 100)
    best_thresh = 0.5
    best_metric = 0
    
    for thresh in thresholds:
        current_metric = calculate_main_metric(y_val, y_proba, thresh)
        if current_metric > best_metric:
            best_metric = current_metric
            best_thresh = thresh
            
    return best_thresh, best_metric

optimal_threshold, best_main_metric = main_metric_optimized_threshold(final_model, X_val, y_val)
print(f"Optimal Threshold: {optimal_threshold:.4f}, Best Main Metric: {best_main_metric:.4f}")


Optimal Threshold: 0.4303, Best Main Metric: 0.6855


In [13]:
def evaluate_full(model, X, y, threshold=0.5):
    y_proba = model.predict_proba(X)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)
    
    metrics = {
        'logloss': log_loss(y, y_proba),
        'auc': roc_auc_score(y, y_proba),
        'f1': f1_score(y, y_pred),
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred),
        'recall': recall_score(y, y_pred),
        'mcc': matthews_corrcoef(y, y_pred)
    }
    
    metrics['main_metric'] = (metrics['accuracy'] + metrics['f1'] + metrics['auc']) / 3
    return metrics

In [14]:
val_metrics = evaluate_full(final_model, X_val, y_val, optimal_threshold)

In [15]:
print("\n" + "="*60)
print(f"{'Metric':<15} {'Value':<10} {'Target Priority':<25}")
print("-"*60)
for metric, value in val_metrics.items():
    priority = ""
    if metric == 'main_metric': priority = "PRIMARY TARGET (Maximize)"
    elif metric in ['accuracy', 'f1', 'auc']: priority = "Main Metric Component"
    print(f"{metric:<15} {value:.6f}   {priority}")
print("="*60)


Metric          Value      Target Priority          
------------------------------------------------------------
logloss         0.616315   
auc             0.718204   Main Metric Component
f1              0.691915   Main Metric Component
accuracy        0.646237   Main Metric Component
precision       0.608771   
recall          0.801364   
mcc             0.309871   
main_metric     0.685452   PRIMARY TARGET (Maximize)


In [16]:
y_pred_val = (final_model.predict_proba(X_val)[:, 1] >= optimal_threshold).astype(int)
cm = confusion_matrix(y_val, y_pred_val)
cm_df = pd.DataFrame(cm, 
                   index=['Actual 0', 'Actual 1'],
                   columns=['Predicted 0', 'Predicted 1'])

print("\nConfusion Matrix (Main-Metric Optimized):")
print(cm_df)
print(f"\nKey Ratios:")
print(f"- True Positive Rate: {cm[1,1]}/{cm[1].sum()} ({cm[1,1]/cm[1].sum():.2%})")
print(f"- True Negative Rate: {cm[0,0]}/{cm[0].sum()} ({cm[0,0]/cm[0].sum():.2%})")
print(f"- Balanced Accuracy: {(cm[1,1]/cm[1].sum() + cm[0,0]/cm[0].sum())/2:.2%}")


Confusion Matrix (Main-Metric Optimized):
          Predicted 0  Predicted 1
Actual 0         1105         1133
Actual 1          437         1763

Key Ratios:
- True Positive Rate: 1763/2200 (80.14%)
- True Negative Rate: 1105/2238 (49.37%)
- Balanced Accuracy: 64.76%


In [17]:
print("\nMain Metric Composition:")
print(f"- Accuracy: {val_metrics['accuracy']:.4f} ({val_metrics['accuracy']/3:.4f})")
print(f"- F1 Score: {val_metrics['f1']:.4f} ({val_metrics['f1']/3:.4f})")
print(f"- AUC:      {val_metrics['auc']:.4f} ({val_metrics['auc']/3:.4f})")
print(f"= Total:   {val_metrics['main_metric']:.4f} (Target)")


Main Metric Composition:
- Accuracy: 0.6462 (0.2154)
- F1 Score: 0.6919 (0.2306)
- AUC:      0.7182 (0.2394)
= Total:   0.6855 (Target)


In [19]:
import json
import joblib

final_model.save_model('../models/HPO3.cbm')

model_metadata = {
    'optimal_threshold': optimal_threshold,
    'performance_metrics': val_metrics,
    'feature_names': X_train.columns.tolist(),
    'categorical_features': list(X_train.select_dtypes(include='object').columns),
    'model_type': 'CatBoostClassifier',
    'model_version': '1.3',
    'training_date': pd.Timestamp.now().strftime("%Y-%m-%d")
}

with open('model_metadata.json', 'w') as f:
    json.dump(model_metadata, f, indent=4)

class ThresholdPredictor:
    def __init__(self, model, threshold):
        self.model = model
        self.threshold = threshold
        
    def predict(self, X):
        proba = self.model.predict_proba(X)[:, 1]
        return (proba >= self.threshold).astype(int)

predictor = ThresholdPredictor(final_model, optimal_threshold)
joblib.dump(predictor, '../models/HPO3_threshold_predictor.joblib')

['../models/HPO3_threshold_predictor.joblib']